In [1]:
import geopandas as gpd
import folium
import warnings
warnings.filterwarnings('ignore')

# Carichiamo tutti i layer
buurten = gpd.read_file(r"C:\Users\andre\OneDrive\Desktop\PARAMETRIC DESIGN\geojson_lnglat_2.json")
gdf = gpd.read_file(r"C:\Users\andre\OneDrive\Desktop\PARAMETRIC DESIGN\geojson_lnglat.json", on_invalid="ignore")
green = gpd.read_file(r"C:\Users\andre\OneDrive\Desktop\PARAMETRIC DESIGN\park_geojson_lnglat.json", on_invalid="ignore")
fietsen = gpd.read_file(r"C:\Users\andre\OneDrive\Desktop\PARAMETRIC DESIGN\fietsnetten_geojson_lnglat.json")
flood = gpd.read_file(r"C:\Users\andre\OneDrive\Desktop\PARAMETRIC DESIGN\overstroming_geojson_lnglat.json")
plans = gpd.read_file(r"C:\Users\andre\OneDrive\Desktop\PARAMETRIC DESIGN\Woningbouwplannen_geojson_lnglat.json", on_invalid="ignore")

print("Tutti i dataset caricati!")
print(f"Quartieri: {len(buurten)}")
print(f"Edifici: {len(gdf)}")
print(f"Verde: {len(green)}")
print(f"Ciclabili: {len(fietsen)}")
print(f"Alluvioni: {len(flood)}")
print(f"Piani abitativi: {len(plans)}")

Tutti i dataset caricati!
Quartieri: 518
Edifici: 44748
Verde: 125
Ciclabili: 8391
Alluvioni: 183
Piani abitativi: 2437


In [2]:
# Creiamo la mappa base centrata su Amsterdam
m = folium.Map(location=[52.3676, 4.9041], 
               zoom_start=12,
               tiles='CartoDB positron')

# --- LAYER 1: Quartieri ---
quartieri_layer = folium.FeatureGroup(name='Neighbourhoods', show=True)
folium.GeoJson(
    buurten.__geo_interface__,
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': '#888888',
        'weight': 0.5,
        'fillOpacity': 0
    }
).add_to(quartieri_layer)
quartieri_layer.add_to(m)

# --- LAYER 2: Aree verdi ---
verde_layer = folium.FeatureGroup(name='Green Areas', show=True)
folium.GeoJson(
    green.__geo_interface__,
    style_function=lambda x: {
        'fillColor': '#2ecc71',
        'color': '#27ae60',
        'weight': 1,
        'fillOpacity': 0.6
    },
    tooltip=folium.GeoJsonTooltip(fields=['Naam', 'Oppervlakte_m2'],
                                   aliases=['Park:', 'Area (m²):'])
).add_to(verde_layer)
verde_layer.add_to(m)

# --- LAYER 3: Rischio alluvioni ---
flood['rischio'] = flood['Kans'].map({
    'Geen significante overstromingskans': 0,
    'Extreem kleine kans: <1/30.000 per jaar': 1,
    'Zeer kleine kans: 1/3.000 tot 1/30.000 per jaar': 2,
    'Kleine kans: 1/300 tot 1/3.000 per jaar': 3
})

colori_flood = {0: '#ffffcc', 1: '#fd8d3c', 2: '#f03b20', 3: '#bd0026'}

flood_layer = folium.FeatureGroup(name='Flood Risk', show=False)
for _, row in flood.iterrows():
    folium.GeoJson(
        row['geometry'].__geo_interface__,
        style_function=lambda x, c=colori_flood.get(row['rischio'], '#ffffcc'): {
            'fillColor': c,
            'color': c,
            'weight': 0.3,
            'fillOpacity': 0.5
        },
        tooltip=f"Risk: {row['Kans']}"
    ).add_to(flood_layer)
flood_layer.add_to(m)

# --- LAYER 4: Piste ciclabili ---
fietsen_layer = folium.FeatureGroup(name='Cycling Network', show=False)
folium.GeoJson(
    fietsen.__geo_interface__,
    style_function=lambda x: {
        'color': '#e74c3c',
        'weight': 1,
        'opacity': 0.6
    }
).add_to(fietsen_layer)
fietsen_layer.add_to(m)

# --- LAYER 5: Piani abitativi futuri ---
futuri = plans[plans['Start_bouw'] > 2025].copy()
piani_layer = folium.FeatureGroup(name='Future Housing (2026–2045)', show=False)
folium.GeoJson(
    futuri.__geo_interface__,
    style_function=lambda x: {
        'fillColor': '#9b59b6',
        'color': '#8e44ad',
        'weight': 1,
        'fillOpacity': 0.7
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['Projectnaam', 'Totaal', 'Start_bouw'],
        aliases=['Project:', 'Units:', 'Start:']
    )
).add_to(piani_layer)
piani_layer.add_to(m)

# Aggiungiamo il controllo dei layer
folium.LayerControl(collapsed=False).add_to(m)

# Salviamo
m.save('amsterdam_dashboard.html')
print("Dashboard salvato!")

Dashboard salvato!
